# Lõi học phase 3: dòng thời gian mọi ghế và ảnh ghép để gán nhãn

Notebook chạy **tầng 1** (model lớn trên mọi ghế, suốt cả video) cho mọi video trong các Dataset đã thêm, tính đặc trưng cửa sổ 3 giây và xuất **ảnh ghép** để Claude gán nhãn. Cuối notebook in bảng thời gian (mốc ≤ 3 × độ dài video).

**Làm lần lượt:**
1. Settings: bật **Internet**, Accelerator **GPU T4 x2**.
2. **Add Input** các Dataset video (Private), ví dụ video-test-rule và video-test-deep.
3. Repo private thì thêm Secret `GITHUB_TOKEN` như các notebook trước.
4. Bấm **Run All**. Tải `timeline.zip` ở tab Output rồi gửi lại (có ảnh thí sinh: chỉ gửi trong project, không đăng công khai).


In [ ]:
import json
import os
import shutil
import subprocess
from pathlib import Path

BRANCH = "claude/project-thread-g4frkm"  # nhánh lõi học phase 3; đổi thành "main" sau khi merge
REPO = "github.com/vuthanhhnguyenn/httm"
WORK = Path("/kaggle/working")
REPO_DIR = WORK / "httm"
OUT_ROOT = WORK / "timeline"

token = ""
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("GITHUB_TOKEN")
except Exception:
    pass  # không có secret: clone bằng URL công khai

def sh(cmd, cwd=None, env=None):
    print("$", cmd)
    try:
        subprocess.run(cmd, shell=True, check=True, cwd=cwd, env=env)
    except subprocess.CalledProcessError as exc:
        # Re-raise without the chained traceback so nothing but the command (never the token) is shown.
        raise RuntimeError(f"Lệnh lỗi (mã {exc.returncode}): {cmd}") from None

print("Python", __import__("sys").version.split()[0])
assert __import__("sys").version_info >= (3, 11), "Cần Python 3.11 trở lên"
VIDEOS = sorted(p for p in Path("/kaggle/input").rglob("*") if p.suffix.lower() in (".mp4", ".avi", ".mov", ".mkv"))
assert VIDEOS, "Không thấy video nào trong các Dataset đã thêm (Add Input)"
for v in VIDEOS:
    print("Video:", v)
sh("nvidia-smi -L")

In [ ]:
# Clone nhánh và cài những gói còn thiếu (Kaggle đã có torch, opencv, numpy). Repo không đóng gói pip nên
# các script tự thêm packages/*/src vào sys.path; không cần pip install -e.
# Token goes only through environment variables (git's GIT_CONFIG_* ), never the URL, argv or .git/config.
env = dict(os.environ)
if token:
    import base64
    basic = base64.b64encode(f"x-access-token:{token}".encode()).decode()
    env.update(GIT_CONFIG_COUNT="1", GIT_CONFIG_KEY_0="http.extraHeader",
               GIT_CONFIG_VALUE_0=f"Authorization: Basic {basic}")
if not REPO_DIR.exists():
    sh(f"git clone --depth 1 --branch {BRANCH} https://{REPO}.git {REPO_DIR}", env=env)
else:
    # Chạy lại trong cùng phiên: lấy commit mới nhất của nhánh thay vì dùng bản cũ đã clone.
    sh(f"git fetch --depth 1 origin {BRANCH}", cwd=REPO_DIR, env=env)
    sh("git reset --hard FETCH_HEAD", cwd=REPO_DIR)
sh('pip install -q "ultralytics==8.4.160" pyyaml "pydantic>=2.9,<3.0" "lightgbm>=4.3,<5.0"')

In [ ]:
# Tải model (kiểm SHA-256 theo models/manifest.yaml)
sh("python scripts/download_model.py", cwd=REPO_DIR)

In [ ]:
# Tầng 1 + đặc trưng + ảnh ghép cho từng video
import torch

GPUS = torch.cuda.device_count()
DEVICES = "--device 0 --device2 1" if GPUS >= 2 else "--device 0"
OUT_ROOT.mkdir(parents=True, exist_ok=True)
errors = {}
for video in VIDEOS:
    out = OUT_ROOT / video.stem
    print(f"\n===== {video.name} =====")
    try:
        sh(f'python scripts/build_timeline.py "{video}" --out "{out}" {DEVICES}', cwd=REPO_DIR)
        sh(f'python scripts/label_sheets.py "{video}" "{out}" --out "{out}/sheets"', cwd=REPO_DIR)
    except Exception as exc:  # noqa: BLE001 - keep going with the next video
        errors[video.name] = str(exc)
        print(f"LỖI {video.name}: {exc}")


In [ ]:
# Bảng thời gian: ratio = tổng thời gian / độ dài video (mốc ≤ 3)
rows = []
for video in VIDEOS:
    out = OUT_ROOT / video.stem
    row = {"video": video.name, "lỗi": errors.get(video.name, "")}
    try:
        t = json.loads((out / "timeline_timing.json").read_text(encoding="utf-8"))
        sheets = len(list((out / "sheets").glob("*.jpg")))
        row.update({"video (s)": t["video_s"], "tổng (s)": t["total_s"], "ratio": t["ratio"], "ghế": t["seats"],
                    "crop": t["crops"], "ảnh ghép": sheets})
    except Exception as exc:  # noqa: BLE001
        row["lỗi"] = row["lỗi"] or str(exc)
    rows.append(row)
for r in rows:
    print(r)


In [ ]:
# Nén để gửi lại (không kèm video): dòng thời gian, tracks, thời gian, đặc trưng, ảnh ghép
pack = WORK / "timeline-gui"
shutil.rmtree(pack, ignore_errors=True)
for video in VIDEOS:
    src, dst = OUT_ROOT / video.stem, pack / video.stem
    dst.mkdir(parents=True, exist_ok=True)
    for file in ("timeline.json.gz", "tracks.jsonl", "timeline_timing.json", "features.csv"):
        if (src / file).is_file():
            shutil.copyfile(src / file, dst / file)
    if (src / "sheets").is_dir():
        shutil.copytree(src / "sheets", dst / "sheets")
archive = shutil.make_archive(str(WORK / "timeline"), "zip", root_dir=WORK, base_dir="timeline-gui")
print(f"Đã nén: {archive} ({Path(archive).stat().st_size / 1e6:.1f} MB). Tải ở tab Output rồi gửi lại.")
